In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder\    #entry point for spark.
        .appName("SparkPractice")\ # label for your job (check ui).
        .master("local[*]")\      # use all cpu cores locally.
        .getOrCreate()


In [ ]:
spark

In [ ]:
data = [
    (1, "2024-01-01", 101, "laptop", 1200, "US"),
    (2, "2024-01-02", 102, "phone", 800, "IN"),
    (3, "2024-01-03", 101, "mouse", 40, "US"),
    (4, "2024-01-04", 103, "keyboard", 100, "UK"),
    (5, "2024-01-05", 104, "monitor", 300, "US"),
    (6, "2024-01-06", 102, "laptop", 1100, "IN"),
    (7, "2024-01-07", 101, "phone", 900, "US")
]

columns = ["order_id", "order_date", "user_id", "product", "amount", "country"]


df = spark.createDataFrame(data,columns)

In [ ]:
df.show()

In [ ]:
df.printSchema()   # no job trigger

In [ ]:
df.describe().show()

In [ ]:
# show only orders where amount > 500
from pyspark.sql.functions import col
df.filter(col("amount")>500).show()



In [ ]:
df.groupBy('country').sum('amount').show()

In [ ]:
from pyspark.sql.types import IntegerType
new_df = df.withColumn('amount',col('amount').cast(IntegerType()))

In [ ]:
new_df.printSchema()

In [ ]:
from pyspark.sql.functions import col, when
new_df.withColumn(
    'amount_category',
    when(col('amount') > 1000, 'High')
    .when((col('amount') <1000) & (col('amount')>500),'Medium')
    .otherwise('Low')
             ).show()


In [ ]:
#TASK 4
#total revenue per user_id and country combination
from pyspark.sql.functions import sum
df.groupBy('user_id','country')\
    .agg(sum("amount").alias("sum"))\
    .show()



In [ ]:
#TASK 5
# Find users with more than 1 order
from pyspark.sql.functions import count
df1 = df.groupBy('user_id').agg(count('order_id').alias('count_of_orders'))

df1.filter(col('count_of_orders')>1).show()

In [ ]:
#TASK 6
#High value users

df1 = df.groupBy('user_id').agg(sum('amount').alias('total_sum'))

df1.filter(col('total_sum')>1500).show()

## NEW DATASET

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder\
        .appName('new_test')\
        .master('local[*]')\
        .getOrCreate()
spark

In [ ]:
data = [
    (1, "login", "2024-01-01", "US"),
    (1, "purchase", "2024-01-02", "US"),
    (2, "login", "2024-01-01", "IN"),
    (2, "logout", "2024-01-01", "IN"),
    (2, "purchase", "2024-01-03", "IN"),
    (3, "login", "2024-01-02", "UK"),
    (3, "purchase", "2024-01-05", "UK"),
    (3, "purchase", "2024-01-06", "UK"),
    (4, "login", "2024-01-01", "US"),
]

columns = ["user_id", "event_type", "event_date", "country"]

events_df = spark.createDataFrame(data,columns)

events_df.printSchema()

In [ ]:
# TASK 7
# total number of events per user

total_events_df = events_df.groupBy(col('user_id')).agg(count(col('event_type')))
total_events_df.show()

In [ ]:
# TASK 8
# users who performed at least 1 purchase

events_df.filter(col("event_type") == "purchase") \
    .select("user_id") \
    .distinct() \
    .show()



In [ ]:
# TASK 9
# Create a column:
# is_active_user

# Logic:

# user has more than 2 events → "YES"

# otherwise → "NO"
from pyspark.sql.functions import *
from pyspark.sql import * 
filter_df = events_df.groupBy('user_id').agg(count(col('event_type')).alias('event_count'))

joined_df = events_df.join(filter_df,on='user_id',how='left')
# joined_df.show()

joined_df.withColumn(
                'is_active_user',
                when(col('event_count')>2,'YES')
                .otherwise('No')
                      ).show()


In [ ]:
dup_data = [
    (1, "login", "2024-01-01"),
    (1, "login", "2024-01-01"),
    (1, "purchase", "2024-01-02"),
    (2, "login", "2024-01-01"),
    (2, "login", "2024-01-01"),
    (3, "purchase", "2024-01-03")
]

dup_columns = ["user_id", "event_type", "event_date"]

dup_df = spark.createDataFrame(dup_data, dup_columns)
# dup_df.show()

#TASK 10
# dup_df.dropDuplicates().show()

#TASK 11
# dup_df.dropDuplicates(["user_id","event_type"]).show()

#TASK 12
from pyspark.sql.functions import count

dup_df.groupBy("user_id", "event_type") \
    .agg(count("*").alias("event_count")) \
    .show()

# Excercise -- Phase 3

In [32]:
from pyspark.sql import SparkSession

In [33]:
spark = SparkSession.builder\
        .appName("Phase_three")\
        .master('local[*]')\
        .getOrCreate()

spark

26/03/20 19:27:02 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [34]:
orders_data = [
    (1, 101, "laptop", 1200, "US", "2024-01-01"),
    (2, 102, "phone", 800, "IN", "2024-01-01"),
    (3, 101, "mouse", 40, "US", "2024-01-02"),
    (4, 103, "keyboard", 100, "UK", "2024-01-02"),
    (5, 104, "monitor", 300, "US", "2024-01-03"),
    (6, 102, "laptop", 1100, "IN", "2024-01-03"),
    (7, 101, "phone", 900, "US", "2024-01-04"),
    (8, 105, "mouse", 35, "US", "2024-01-04"),
    (9, 105, "mouse", 35, "US", "2024-01-04"),
    (10, 106, "tablet", 600, "CA", "2024-01-05")
]

orders_columns = ["order_id", "user_id", "product", "amount", "country", "order_date"]

orders_df = spark.createDataFrame(orders_data, orders_columns)


In [43]:
from pyspark.sql.functions import col,count
orders_df.show()

+--------+-------+--------+------+-------+----------+
|order_id|user_id| product|amount|country|order_date|
+--------+-------+--------+------+-------+----------+
|       1|    101|  laptop|  1200|     US|2024-01-01|
|       2|    102|   phone|   800|     IN|2024-01-01|
|       3|    101|   mouse|    40|     US|2024-01-02|
|       4|    103|keyboard|   100|     UK|2024-01-02|
|       5|    104| monitor|   300|     US|2024-01-03|
|       6|    102|  laptop|  1100|     IN|2024-01-03|
|       7|    101|   phone|   900|     US|2024-01-04|
|       8|    105|   mouse|    35|     US|2024-01-04|
|       9|    105|   mouse|    35|     US|2024-01-04|
|      10|    106|  tablet|   600|     CA|2024-01-05|
+--------+-------+--------+------+-------+----------+



In [63]:
# Questions
# a) Basic

# Show only the rows where country = 'US'.
# orders_df.filter(col('country')=='US').show()

# Select only user_id, product, and amount.
# orders_df.select(col('user_id'),col('product'),col('amount')).show()

# b) Medium

# Find total amount spent per user_id.
# orders_df.groupBy('user_id').agg(sum('amount').alias('total_amount_spent')).show()

# Find users who placed more than 1 order.
# orders_placed = orders_df.groupBy('user_id').agg(count('product').alias('total_count_user'))
# orders_placed.filter(col('total_count_user')>1).select(col('user_id')).show()

# Create a new column amount_band with:
# "HIGH" if amount > 1000
# "MEDIUM" if amount is between 500 and 1000 inclusive
# "LOW" otherwise

# orders_df.withColumn(
#     'amount_band',
#     when(col('amount')>1000,'HIGH')
#     .when((col('amount') >500) & (col('amount')<1000),'MEDIUM')
#     .otherwise('LOW')
# ).show()

# c) HARD - FAANG LEVELLED

# Create a row-level column is_repeat_user such that:
# "YES" if the user has placed more than 1 order overall
# "NO" otherwise

# filtered_df = orders_df.groupBy('user_id').agg(count('product').alias('cts'))

# joined_df = orders_df.join(filtered_df, on='user_id',how='left')
# joined_df.show()

# joined_df.withColumn('is_repeat_user',
#                      when(col('cts')>1,"YES")
#                      .otherwise("NO")
#                     ).show()

# Find unique users who purchased either "laptop" or "phone".
# filterd_df = orders_df.filter((col('product') == 'laptop') | (col('product') == 'phone'))
# unique_df = filterd_df.dropDuplicates(['product']).show()

# Count how many times each (user_id, product) combination appears, then show only combinations where the count is greater than 1.
filtrd_df = orders_df.groupBy('user_id','product').agg(count('*').alias('cnt'))
filtrd_df.filter(col('cnt')>1)\
         .select('user_id').show()



+-------+
|user_id|
+-------+
|    105|
+-------+

